# 00 — 建立 PVC 持久化 Python 環境

對應 [Jupyter on OpenShift Part 5: Ad-hoc Package Installation](https://www.redhat.com/en/blog/jupyter-on-openshift-part-5-ad-hoc-package-installation) 的現代做法。

將 Python 虛擬環境建立在 PVC（`/opt/app-root/src/venv`），讓 ad-hoc 安裝的套件在 Workbench 重啟後仍可恢復。

**請先執行本 Notebook，再執行訓練 Notebook。**

> **離線／自訂映像**：若講師已提供含套件的 Workbench 映像（套件已 bake-in），可**略過本 notebook 的 pip**，直接跑 `01`。僅在需要額外 ad-hoc 套件時才建 venv（並改指向內網 PyPI 或 `--find-links` wheels）。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/opt/app-root/src")
VENV_DIR = WORKDIR / "venv"
REQUIREMENTS = WORKDIR / "requirements.txt"

print(f"WORKDIR (PVC): {WORKDIR}")
print(f"VENV: {VENV_DIR}")
print(f"Python: {sys.version}")

In [ ]:
if not VENV_DIR.exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
    print(f"Created venv at {VENV_DIR}")
else:
    print(f"venv already exists at {VENV_DIR}")

In [ ]:
pip = str(VENV_DIR / "bin" / "pip")
python = str(VENV_DIR / "bin" / "python")

subprocess.run([pip, "install", "--upgrade", "pip"], check=True)
subprocess.run([pip, "install", "ipykernel", "scikit-learn", "joblib", "pandas", "numpy", "requests"], check=True)
subprocess.run([
    python, "-m", "ipykernel", "install", "--user",
    "--name=dev-venv", "--display-name=Python (dev-venv)",
], check=True)
print("Kernel 'Python (dev-venv)' registered.")

In [ ]:
subprocess.run([pip, "freeze"], check=True)
result = subprocess.run([pip, "freeze"], capture_output=True, text=True, check=True)
REQUIREMENTS.write_text(result.stdout)
print(f"Saved {REQUIREMENTS}")

## 下一步

1. 在 Jupyter 右上角將 Kernel 切換為 **Python (dev-venv)**
2. 日常 ad-hoc 安裝：
   ```bash
   source /opt/app-root/src/venv/bin/activate
   pip install <package>
   pip freeze > /opt/app-root/src/requirements.txt
   ```
3. Workbench 重啟後恢復：
   ```bash
   source /opt/app-root/src/venv/bin/activate && pip install -r /opt/app-root/src/requirements.txt
   ```
4. 執行 `01-train-sklearn-iris.ipynb` 進行訓練

建立 Workbench／專案步驟見 [getting-started-ui-tutorial.ipynb](../docs-ipynb/getting-started-ui-tutorial.ipynb)。
